# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/Leem1nwon/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.resnet import resnet18
from src.models.vit import vit_small_patch16_224   # best 백본 (Level 2 ImageNet-pretrained)

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# Colab `Run All` 재현 시 wandb 는 비활성화합니다 (조교 채점 환경 = WANDB_DISABLED=true).
# 본인이 직접 학습하며 로깅하려면 이 줄을 주석 처리하고 아래에서 RUN_TRAINING=True, wandb.login() 을 사용하세요.
import os
os.environ["WANDB_DISABLED"] = "true"

# ===== 재현 모드 스위치 =====
# RUN_TRAINING = False → 학습을 건너뛰고 사전학습 체크포인트(.pth)를 로드해 eval/그림만 재현 (조교 채점 경로).
# RUN_TRAINING = True  → 처음부터 학습 (H100 권장; T4 에서는 매우 오래 걸립니다).
RUN_TRAINING = False

WANDB_PROJECT = "aue8088-pa2" if RUN_TRAINING else None   # 재현 모드에서는 wandb 끔
WANDB_TAGS    = ["level3"]
EXPERIMENT_NAME = "mixup-cutmix"   # best 기법

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

# 옵션 A — 가장 불균형이 심한 weather 속성 기준 class-balanced sampler 사용
sampler = class_balanced_sampler(train_ds, attribute="weather")
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
# --- 사전학습 체크포인트 다운로드 (재현 모드 RUN_TRAINING=False) ----------------
# checkpoints/ 에 level3_best.pth 가 없으면 Google Drive 에서 체크포인트 zip 을 받아 압축 해제.
# (제출 전 체크포인트 zip 을 본인 드라이브에 올린 뒤 아래 ID 를 교체하세요.)
CKPT_DIR = "../checkpoints"
GDRIVE_CKPT_ID = "PUT_CHECKPOINT_ZIP_ID_HERE"   # ← 체크포인트 zip 의 Google Drive 파일 ID

if not RUN_TRAINING and not os.path.exists(f"{CKPT_DIR}/level3_best.pth"):
    if GDRIVE_CKPT_ID == "PUT_CHECKPOINT_ZIP_ID_HERE":
        print("[주의] GDRIVE_CKPT_ID 를 체크포인트 zip 파일 ID 로 교체하세요.")
    else:
        import gdown
        gdown.download(id=GDRIVE_CKPT_ID, output="../checkpoints.zip", quiet=False)
        with zipfile.ZipFile("../checkpoints.zip") as zf:
            zf.extractall("..")
        print("체크포인트 다운로드/해제 완료")
else:
    print(f"체크포인트 준비 상태: RUN_TRAINING={RUN_TRAINING}")

In [ ]:
# ===== best 모델 준비 (학습 또는 체크포인트 로드) =====
# best 기법 = mixup-cutmix (plain CE, sampler 없음). 백본 = ViT-S/16 (ImageNet-pretrained).
#
# 알고리즘 순서:
#   RUN_TRAINING=True  : ViT 빌드 → AdamW/Cosine → mixup/cutmix 학습 루프 → level3_best.pth 저장
#   RUN_TRAINING=False : ViT 빌드 → level3_best.pth state_dict 로드 → eval 모드
#
# (아래 loss_fns 는 "속성별 다른 loss" 옵션을 보여주기 위한 참고 정의입니다. best 설정은 plain CE.)
samples_s = train_ds.class_counts("scene")
loss_fns_perattr = {
    "weather":   FocalLoss(gamma=2.0).to(device),
    "scene":     ClassBalancedLoss(samples_s).to(device),
    "timeofday": nn.CrossEntropyLoss(),
}
# best 설정(mixup-cutmix)에서 실제로 사용한 loss = plain CrossEntropy (3 속성 동일).
loss_fns = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

epochs = 25
model = vit_small_patch16_224().to(device)

if not RUN_TRAINING:
    # --- 재현 경로: best 체크포인트 로드 -------------------------------------
    ckpt = torch.load(f"{CKPT_DIR}/level3_best.pth", map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()
    print("level3_best.pth 로드 완료 (mixup-cutmix / ViT-S/16)")

In [ ]:
# ===== 학습 루프 (RUN_TRAINING=True 일 때만 실행) =====
# best 기법 = Mixup/CutMix. 매 step 50% 확률로 Mixup, 50% 확률로 CutMix 를 적용하고
# mixed_loss 로 3-head 라벨 혼합 손실을 계산합니다. (재현 모드에서는 통째로 skip)
from tqdm import tqdm

def step_with_mix(images, targets):
    """50% 확률로 Mixup, 나머지 50% 확률로 CutMix 적용."""
    if torch.rand(1).item() < 0.5:
        x, ya, yb, lam = mixup_data(images, targets, alpha=0.2)
    else:
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam)

if RUN_TRAINING:
    from src.utils.metrics import average_macro_f1

    # best 설정: mixup-cutmix 는 sampler 없이 shuffle 로더 사용
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader_mix = DataLoader(
        train_ds, batch_size=BATCH, shuffle=True,
        num_workers=2, worker_init_fn=seed_worker, generator=g, pin_memory=True,
    )

    optim = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    logger = WandbLogger(
        project=WANDB_PROJECT,
        run_name=f"level3-{EXPERIMENT_NAME}",
        config={"backbone": "vit_small_patch16_224", "aug": "mixup+cutmix",
                "loss": "plain_ce", "epochs": epochs, "batch": BATCH,
                "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED},
        tags=WANDB_TAGS + [EXPERIMENT_NAME],
    )
    helper = MultiTaskTrainer(model, optim, sched, loss_fns, device,
                              TrainConfig(epochs=epochs), logger=logger)

    history = {"train_loss": [], "val_avg_mf1": []}
    best_mf1 = -1.0
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for batch in tqdm(train_loader_mix, desc=f"epoch {epoch+1}/{epochs}"):
            images = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}
            optim.zero_grad()
            loss = step_with_mix(images, targets)
            loss.backward()
            optim.step()
            running += loss.item()
        sched.step()
        train_loss = running / max(1, len(train_loader_mix))

        val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
        val_mf1 = average_macro_f1(val_pred, val_tgt)
        history["train_loss"].append(train_loss)
        history["val_avg_mf1"].append(val_mf1)
        logger.log({"train_loss": train_loss, "val_avg_mf1": val_mf1}, step=epoch)
        print(f"[epoch {epoch+1}] train_loss={train_loss:.4f}  val_avg_mf1={val_mf1:.4f}")

        if val_mf1 > best_mf1:
            best_mf1 = val_mf1
            os.makedirs(CKPT_DIR, exist_ok=True)
            torch.save({"state_dict": model.float().state_dict(), "history": history,
                        "best_val_avg_mf1": best_mf1, "seed": SEED},
                       f"{CKPT_DIR}/level3_best.pth")

    logger.finish()
    # 최종적으로 best 체크포인트를 다시 로드해 eval 상태로 둠
    ckpt = torch.load(f"{CKPT_DIR}/level3_best.pth", map_location="cpu")
    model.load_state_dict(ckpt["state_dict"]); model.to(device).eval()
    print(f"학습 완료 — best val_avg_mf1={best_mf1:.4f}")
else:
    print("RUN_TRAINING=False → 학습 루프 skip (체크포인트로 eval).")

In [ ]:
# 학습을 직접 수행한 경우(RUN_TRAINING=True) 속성별 confusion matrix + per-class F1 표를 wandb 에 업로드.
# 재현 모드(RUN_TRAINING=False)에서는 아래 분석 셀에서 직접 그림/표를 그립니다.
if RUN_TRAINING:
    val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
    cms = confusion_matrices(val_pred, val_tgt)
    prf = per_class_prf(val_pred, val_tgt)
    for a in ATTRIBUTES:
        logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
        rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
        logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
    logger.finish()

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.

In [ ]:
# (a) 12-run ablation 비교 표 — 저장된 메트릭에서 로드 (재학습 X)
# tables/level3_all_metrics.json: 12개 기법 각각 name / best_val_avg_mf1 / final_per_mf1 보유.
import json

with open("../tables/level3_all_metrics.json") as f:
    all_runs = json.load(f)
all_runs = sorted(all_runs, key=lambda r: r["best_val_avg_mf1"], reverse=True)

print(f"{'technique':28s} {'best Avg-MF1':>12s}  {'weather':>8s} {'scene':>7s} {'time':>7s}")
print("-" * 70)
for r in all_runs:
    per = r["final_per_mf1"]
    star = "  ★best" if r["stem"] == "level3_best" or r["name"] == "mixup-cutmix" else ""
    print(f"{r['name']:28s} {r['best_val_avg_mf1']:12.4f}  "
          f"{per['weather']:8.3f} {per['scene']:7.3f} {per['timeofday']:7.3f}{star}")

In [ ]:
# (b) weather 속성 per-class F1 — baseline vs best(mixup-cutmix) 소수/다수 클래스 비교
# 저장된 prf 에서 weather per-class F1 을 읽어 비교 (수치 지어내지 않음).
runs_by_stem = {r["stem"]: r for r in all_runs}
base = runs_by_stem["level3_baseline"]["prf"]["weather"]
best = runs_by_stem["level3_mixup_cutmix"]["prf"]["weather"]

print(f"{'weather class':16s} {'support':>7s} {'F1(baseline)':>13s} {'F1(best)':>10s} {'Δ':>8s}")
print("-" * 60)
for cls, sup, f_b, f_m in zip(base["class"], base["support"], base["f1"], best["f1"]):
    tag = "  (minority)" if sup <= 50 else ""
    print(f"{cls:16s} {sup:7d} {f_b:13.3f} {f_m:10.3f} {f_m - f_b:+8.3f}{tag}")
print("\n* foggy(support=0) 는 train 에 0장 → F1=0 (학습 불가, 두 모델 공통). "
      "snowy/rainy/partly cloudy 등 소수 클래스의 변화에 주목.")

In [ ]:
# (c) best 모델(level3_best.pth, mixup-cutmix) 속성별 정규화 Confusion Matrix
# 로드된 model 로 직접 추론해 confusion matrix 를 계산 (메트릭 함수 사용).
import matplotlib.pyplot as plt
import seaborn as sns

preds, _, tgts, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(preds, tgts)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax_, a in zip(axes, ATTRIBUTES):
    sns.heatmap(cms[a], annot=True, fmt=".2f", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES[a], yticklabels=CLASS_NAMES[a], ax=ax_)
    ax_.set_title(f"best (mixup-cutmix) — {a}")
    ax_.set_xlabel("pred"); ax_.set_ylabel("true")
plt.tight_layout(); plt.show()

from src.utils.metrics import average_macro_f1, per_attribute_macro_f1
per = per_attribute_macro_f1(preds, tgts)
print(f"best 모델 val Avg-MF1 = {average_macro_f1(preds, tgts):.4f}  "
      f"(weather={per['weather']:.3f}, scene={per['scene']:.3f}, timeofday={per['timeofday']:.3f})")